# 12. The model explorer

Navigate a model like a workspace: a searchable tree over the owning
structure, a diagram pane that follows your selection, and two-way
linkage between them.

**You will learn how to:**

- open the explorer (`explorer.explore`) and navigate the tree
  (keyboard, search, kind badges);
- switch between the diagram kinds that apply to a selection
  (structure, state, action, requirements);
- follow selection both ways: tree -> diagram highlight and diagram
  click -> tree reveal;
- dock the explorer into the JupyterLab shell (`layout="lab"`, the
  `[explorer]` extra).


`longeron.explorer.explore(model)` puts a **tree navigator** over the model's
owning structure beside a **diagram pane** that renders the selected element
through whichever diagram kinds apply to it (structure / state / action /
requirements).

- Click a tree row (or use the arrow keys + Enter) to select an element; the
  diagram pane renders it and highlights it.
- Click a node **in the diagram** to select and reveal it in the tree.
- The kind switcher only offers the kinds that apply to the selection, and
  switching preserves the selection.
- The tree's filter box live-prunes the tree to matches + ancestors with a
  `matches/total` count; each diagram keeps its own toolbar (fit / center /
  routing / search).

This notebook is a demo **and** a self-check: every section asserts what it
demonstrates, so executing it end-to-end is a test.

In [ ]:
from pathlib import Path

import longeron
from longeron import explorer

# resolve the repo root regardless of where jupyter was started
here = Path.cwd().resolve()
ROOT = next(p for p in (here, *here.parents) if (p / "examples" / "drone.sysml").exists())

drone = longeron.load(str(ROOT / "examples" / "drone.sysml"))
ex = explorer.explore(drone)  # layout="auto": docks into JupyterLab when
# ipylab + a Lab frontend are available, else renders inline (as here)
print("layout strategy:", ex.layout_strategy)
assert isinstance(ex.tree, explorer.TreeView)  # the swappable tree seam
assert ex.tree.total_count == 46
assert ex.kind == "structure"
ex

## Tree → diagram

Selecting a tree element renders the applicable diagram and highlights the
element through the diagram's own selection tool. Diagrams are cached per
(scope, kind), so re-selecting inside the same package reuses the same widget.

In [ ]:
ex.select("Drone::QuadCopter")
assert ex.element is drone.find("Drone::QuadCopter")
assert tuple(ex.kind_switcher.options) == ("structure", "requirements")
assert tuple(ex.diagram.view.selection.ids) == ("Drone::QuadCopter",)

structure = ex.diagram
ex.select("Drone::Battery")
assert ex.diagram is structure  # same package scope: cached widget reused
print("tree -> diagram ok:", ex.tree.selected)

## Diagram → tree, with no selection echo

Setting the diagram's selection (what a browser click does) selects and
reveals the element in the tree **without rebuilding the clicked diagram**,
and every trait settles after a single write — the echo dies at its first
fixpoint.

In [ ]:
widget = ex.diagram
tree_writes, diagram_writes = [], []
ex.tree.observe(lambda ch: tree_writes.append(ch["new"]), "selected")
widget.view.selection.observe(lambda ch: diagram_writes.append(ch["new"]), "ids")

widget.view.selection.ids = ["Drone::PlanBattery"]  # a diagram click
assert ex.tree.selected == ["Drone::PlanBattery"]
assert "action" in ex.kind_switcher.options
assert ex.diagram is widget  # not rebuilt
assert tree_writes == [["Drone::PlanBattery"]]  # exactly one write each way
assert diagram_writes == [("Drone::PlanBattery",)]
print("diagram -> tree ok, no echo")

## Kind switching preserves the selection

State and action elements offer their behavioral views; a nested state scopes
to its whole machine with the selection highlighted.

In [ ]:
ex.select("Drone::FlightStates::idle")
assert tuple(ex.kind_switcher.options) == ("structure", "state", "requirements")
ex.kind = "state"
assert ex.tree.selected == ["Drone::FlightStates::idle"]
assert tuple(ex.diagram.view.selection.ids) == ("Drone::FlightStates::idle",)
ex.diagram

## The requirements view

A structure-view projection of the owning package's requirement definitions /
usages, satisfy usages and their satisfying elements — satisfy keyword edges
included — without touching the real model (elements are listed, never
re-parented).

In [ ]:
from longeron.explorer import _diagram_node_ids

reqs = longeron.loads("""
package P {
    requirement def SafeMass;
    requirement massReq : SafeMass;
    part sys;
    satisfy massReq by sys;
}
""")
rex = explorer.explore(reqs)
rex.select("P::massReq")
rex.kind = "requirements"
ids = _diagram_node_ids(rex.diagram)
assert {"P::SafeMass", "P::massReq", "P::sys"} <= ids
satisfies = [
    e for e in rex.diagram.source.value.edges if "sysml-edge-satisfies" in e.properties.cssClasses
]
assert [(e.source.id, e.target.id) for e in satisfies] == [("P::sys", "P::massReq")]
assert reqs.find("P::sys").owner is reqs.find("P")  # model untouched
rex

## The big one

`uav_missions.sysml` is the largest shipped example. The tree ships its whole
structure but renders rows lazily (only expanded rows reach the DOM), and the
kernel-side filter mirrors the browser's `matches/total` count.

In [ ]:
uav = longeron.load(str(ROOT / "examples" / "uav_missions.sysml"))
uex = explorer.explore(uav)
assert uex.tree.total_count > 400

uex.tree.query = "battery"
assert uex.tree.match_count > 0
print(f"filter: {uex.tree.match_count}/{uex.tree.total_count} match 'battery'")
uex.tree.query = ""

uex.select("UavMissions::MissionRequirements")
assert "requirements" in uex.kind_switcher.options
uex.kind = "requirements"
uex